# 1kg_eur - rare-variant PCA: loadings, PC and correlation plots

Pure visualisation. No fitting, no plink2 - reads the `.eigenvec`,
`.eigenval` and `.eigenvec.allele` files every earlier notebook left behind and
puts them on common axes.

**Discovers** whatever exists under `~/scratch_1kg_eur_rare/`:

| directory | from | fits |
|---|---|---|
| `unrel/` | `rare_pca_unrelated.ipynb` | arm x {all, unrel}, full genome |
| `oddeven/` | `rare_pca_oddeven.ipynb` | arm x sample set x {odd, even} |
| `fractionate/` | `rare_pca_fractionate.ipynb` | arm x MAF bin |

Missing directories are skipped, so this is useful after any subset of them.

### What each plot is for

1. **Scree** - eigenvalue spectrum per fit, on shared axes. A long flat tail
   means the PCs past the first few are noise.
2. **Loadings, Manhattan style** - |weight| against true genomic position with
   chromosomes coloured alternately. This is the plot that shows whether a PC is
   carried by a region: a vertical stripe over one locus. Plotting against
   *position* rather than variant index is the point - index hides how wide the
   implicated interval is.
3. **Top-loading table and zoom** - the strongest region per PC, with a
   position-zoomed panel so you can see whether it is one variant or a block.
4. **PC scatters** - hexbin with log counts, because 223K points overplot into a
   solid blob as a scatter. Family/outlier structure shows as isolated low-count
   cells far from the mass, which a scatter with alpha cannot show.
5. **Cross-fit PC correlation** - heatmaps of |r| between the PCs of any two
   fits. Off-diagonal structure means the axes are the same but reordered.
6. **Within-fit orthogonality** - should be the identity by construction; a
   sanity check that the files are what we think.
7. **Extreme individuals** - who sits furthest out, and whether the same people
   are extreme in several fits.

## Config and discovery

In [ ]:
import os, glob, re, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

SAMPLE_SET = "1kg_eur"
WS_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
OUT_GS = f"{WS_GS}/{SAMPLE_SET}/01_ancestry/rare_pca_plots"

RARE = os.path.expanduser(f"~/scratch_{SAMPLE_SET}_rare")
OUT  = f"{RARE}/plots"
os.makedirs(OUT, exist_ok=True)

N_PCS   = 10
N_SHOW  = 5     # PCs to draw in the per-PC panels

fits = {}
for d in ("unrel", "oddeven", "fractionate"):
    for ev in sorted(glob.glob(f"{RARE}/{d}/*.eigenvec")):
        stem = ev[: -len(".eigenvec")]
        fits[f"{d}/{os.path.basename(stem)}"] = stem

print(f"{len(fits)} fits found:")
for k in fits:
    has_w = os.path.isfile(f"{fits[k]}.eigenvec.allele")
    print(f"  {k:46s}  loadings: {'yes' if has_w else 'no'}")

# Per-fit panels are one figure each, so default to the core comparison - both
# arms' full-genome fits from unrel/ - rather than every fit found. The scree and
# correlation sections always use everything.
#   FITS_PANEL = list(fits)                      # all of them
#   FITS_PANEL = [k for k in fits if "b" in k]   # just the MAF bins
FITS_PANEL = [k for k in fits if k.startswith("unrel/")] or list(fits)
print(f"\nper-fit panels for {len(FITS_PANEL)} fits (edit FITS_PANEL to change):")
for k in FITS_PANEL:
    print("  " + k)

## Load

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS + 1)]

def read_eigenvec(stem):
    d = pd.read_csv(f"{stem}.eigenvec", sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: "person_id"})
    d["person_id"] = d["person_id"].astype(str)
    cols = [c for c in PC_COLS if c in d.columns]
    return d.set_index("person_id")[cols]

def read_weights(stem):
    p = f"{stem}.eigenvec.allele"
    if not os.path.isfile(p):
        return None
    L = pd.read_csv(p, sep=r"\s+")
    idc = "#ID" if "#ID" in L.columns else "ID"
    parts = L[idc].str.split(":", n=2, expand=True)
    L = L.rename(columns={idc: "ID"})
    L["CHROM"] = (parts[0].astype(str).str.replace("^chr", "", regex=True))
    L["POS"] = pd.to_numeric(parts[1], errors="coerce")
    L = L[L["CHROM"].str.fullmatch(r"\d+")].dropna(subset=["POS"])
    L["CHROM_N"] = L["CHROM"].astype(int)
    return L.sort_values(["CHROM_N", "POS"]).reset_index(drop=True)

E, V, W = {}, {}, {}
for k, stem in fits.items():
    E[k] = read_eigenvec(stem)
    ev = np.loadtxt(f"{stem}.eigenval")
    V[k] = np.atleast_1d(ev) / np.atleast_1d(ev).sum() * 100
    w = read_weights(stem)
    if w is not None:
        W[k] = w
    print(f"{k:46s} n={len(E[k]):>7,}  pcs={E[k].shape[1]}  "
          f"variants={len(w) if w is not None else 0:>7,}")

## 1. Scree

Shared y-axis so fits are comparable. A fit whose PC1 barely exceeds PC10 has no
dominant axis at all.

In [ ]:
groups = {}
for k in fits:
    groups.setdefault(k.split("/")[0], []).append(k)

fig, axes = plt.subplots(1, len(groups), figsize=(6 * len(groups), 4.2), squeeze=False)
for ax, (g, keys) in zip(axes[0], groups.items()):
    for k in keys:
        ax.plot(range(1, len(V[k]) + 1), V[k], "-o", ms=3,
                label=k.split("/", 1)[1])
    ax.set_xlabel("PC"); ax.set_ylabel("% variance explained")
    ax.set_title(g); ax.set_xticks(range(1, N_PCS + 1))
    ax.legend(fontsize=6)
plt.suptitle("Eigenvalue spectra")
plt.tight_layout(); plt.savefig(f"{OUT}/01_scree.png", dpi=150); plt.show()

## 2. Loadings, Manhattan style

|weight| against genomic position, chromosomes alternating in colour. A PC
tracking genome-wide structure looks like uniform noise across all 22; a PC
tracking a locus shows a spike.

In [ ]:
def manhattan(ax, L, col):
    off, ticks, labels = 0.0, [], []
    for c in sorted(L["CHROM_N"].unique()):
        g = L[L["CHROM_N"] == c]
        x = off + g["POS"].to_numpy()
        ax.scatter(x, g[col].abs(), s=0.4, alpha=0.5, rasterized=True,
                   color="0.25" if c % 2 else "tab:blue")
        ticks.append(off + g["POS"].mean()); labels.append(str(c))
        off += g["POS"].max() + 2e7
    ax.set_xticks(ticks); ax.set_xticklabels(labels, fontsize=6)
    ax.set_xlim(0, off); ax.set_ylabel(col)
    return ax

for k in FITS_PANEL:
    if k not in W:
        continue
    L = W[k]
    npc = min(N_SHOW, sum(1 for c in PC_COLS if c in L.columns))
    fig, axes = plt.subplots(npc, 1, figsize=(15, 1.9 * npc), sharex=True)
    for i, ax in enumerate(np.atleast_1d(axes), 1):
        manhattan(ax, L, f"PC{i}")
    plt.suptitle(f"{k} - |allele weight| by position ({len(L):,} variants)")
    plt.tight_layout()
    plt.savefig(f"{OUT}/02_loadings_{k.replace('/', '_')}.png", dpi=150)
    plt.show()

## 3. Where the top loadings sit, and how wide

For each PC, the strongest variant and a +-2 Mb zoom around it. One isolated
point means a single variant; a cluster means an LD block or a region.

In [ ]:
rows = []
for k in FITS_PANEL:
    if k not in W:
        continue
    L = W[k]
    for i in range(1, min(N_SHOW, 6) + 1):
        col = f"PC{i}"
        if col not in L.columns:
            continue
        t = L.reindex(L[col].abs().sort_values(ascending=False).index).head(5)
        for rank, r in enumerate(t.itertuples(), 1):
            rows.append({"fit": k, "PC": i, "rank": rank, "ID": r.ID,
                         "chrom": r.CHROM_N, "pos_mb": r.POS / 1e6,
                         "abs_weight": abs(getattr(r, col))})
top = pd.DataFrame(rows)
print(top.round(4).to_string(index=False))

In [ ]:
for k in FITS_PANEL:
    if k not in W:
        continue
    L, npc = W[k], min(N_SHOW, 5)
    fig, axes = plt.subplots(1, npc, figsize=(3.4 * npc, 3.2), squeeze=False)
    for i, ax in zip(range(1, npc + 1), axes[0]):
        col = f"PC{i}"
        if col not in L.columns:
            continue
        best = L.loc[L[col].abs().idxmax()]
        w = L[(L["CHROM_N"] == best["CHROM_N"])
              & (L["POS"] - best["POS"]).abs().le(2e6)]
        ax.scatter(w["POS"] / 1e6, w[col].abs(), s=8, alpha=0.7)
        ax.axvline(best["POS"] / 1e6, color="crimson", ls="--", lw=1)
        ax.set_title(f"{col}: chr{int(best['CHROM_N'])}\n{len(w)} variants in 4 Mb",
                     fontsize=8)
        ax.set_xlabel("Mb"); ax.set_ylabel("|weight|")
    plt.suptitle(f"{k} - zoom on the strongest locus per PC", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{OUT}/03_zoom_{k.replace('/', '_')}.png", dpi=150)
    plt.show()

## 4. PC scatters

Hexbin with log counts. At 223K individuals a scatter is a solid blob; hexbin
keeps the low-count cells visible, which is where family clusters and outliers
live.

In [ ]:
PAIRS = [(1, 2), (3, 4), (5, 6), (1, 3)]
for k in FITS_PANEL:
    d = E[k]
    pairs = [(i, j) for i, j in PAIRS
             if f"PC{i}" in d.columns and f"PC{j}" in d.columns]
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.1 * len(pairs), 3.8),
                             squeeze=False)
    for (i, j), ax in zip(pairs, axes[0]):
        hb = ax.hexbin(d[f"PC{i}"], d[f"PC{j}"], gridsize=110, norm=LogNorm(),
                       cmap="viridis", mincnt=1)
        ax.set_xlabel(f"PC{i}"); ax.set_ylabel(f"PC{j}")
        fig.colorbar(hb, ax=ax, fraction=0.04, label="individuals")
    plt.suptitle(f"{k}  (n={len(d):,})", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{OUT}/04_pcpairs_{k.replace('/', '_')}.png", dpi=150)
    plt.show()

## 5. Cross-fit PC correlations

|r| between every PC of one fit and every PC of another, over the individuals
they share. Read the diagonal for "same PC, same order"; bright off-diagonal
cells mean the same axis came out in a different position, which is expected
whenever eigenvalues are close.

In [ ]:
def cross(k1, k2):
    a, b = E[k1], E[k2]
    ids = a.index.intersection(b.index)
    ca = [c for c in PC_COLS if c in a.columns]
    cb = [c for c in PC_COLS if c in b.columns]
    A = a.loc[ids, ca].to_numpy(); B = b.loc[ids, cb].to_numpy()
    R = np.corrcoef(A.T, B.T)[:len(ca), len(ca):]
    return np.abs(R), ca, cb, len(ids)

def plot_cross(ax, k1, k2):
    R, ca, cb, n = cross(k1, k2)
    im = ax.imshow(R, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks(range(len(cb))); ax.set_xticklabels(range(1, len(cb) + 1), fontsize=6)
    ax.set_yticks(range(len(ca))); ax.set_yticklabels(range(1, len(ca) + 1), fontsize=6)
    ax.set_xlabel(k2.split("/", 1)[1], fontsize=7)
    ax.set_ylabel(k1.split("/", 1)[1], fontsize=7)
    ax.set_title(f"n={n:,}", fontsize=7)
    return im

# every pair worth comparing: same arm, different fit
def arm_of(k):
    m = re.search(r"(nonhm3_rare_all|hm3_rare_all)", k)
    return m.group(1) if m else None

pairs = []
for arm in ("nonhm3_rare_all", "hm3_rare_all"):
    ks = [k for k in fits if arm_of(k) == arm]
    base = next((k for k in ks if k.startswith("unrel/") and k.endswith("_all")), None)
    if base is None:
        continue
    for k in ks:
        if k != base:
            pairs.append((base, k))

if pairs:
    ncol = 4
    nrow = int(np.ceil(len(pairs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.1 * nrow),
                             squeeze=False)
    for ax, (k1, k2) in zip(axes.flat, pairs):
        im = plot_cross(ax, k1, k2)
    for ax in axes.flat[len(pairs):]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, fraction=0.015, label="|r|")
    plt.suptitle("Cross-fit |r| - each panel vs that arm's full-genome all-sample fit")
    plt.savefig(f"{OUT}/05_cross_fit_corr.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("no comparable pairs found")

In [ ]:
# the same information as a table: best match and its |r| per PC
rows = []
for k1, k2 in pairs:
    R, ca, cb, n = cross(k1, k2)
    for i in range(len(ca)):
        rows.append({"reference": k1, "other": k2, "PC": i + 1,
                     "diag_abs_r": R[i, i] if i < R.shape[1] else np.nan,
                     "max_abs_r": R[i].max(),
                     "best_match": int(R[i].argmax()) + 1, "n": n})
xf = pd.DataFrame(rows)
if len(xf):
    print(xf.pivot_table(index="PC", columns="other", values="max_abs_r")
          .round(3).to_string())

## 6. Within-fit orthogonality

|r| between PCs of the *same* fit. This must be the identity - PCs are
orthogonal by construction. Anything bright off the diagonal means the file is
not what we think it is, so this is a check on the inputs rather than a result.

In [ ]:
worst = 0.0
for k in fits:
    d = E[k]
    R = np.abs(np.corrcoef(d.to_numpy().T))
    np.fill_diagonal(R, 0)
    worst = max(worst, R.max())
    if R.max() > 0.05:
        print(f"WARNING {k}: max off-diagonal |r| = {R.max():.4f}")
print(f"max off-diagonal |r| across all {len(fits)} fits: {worst:.2e}")
print("(expected ~0; PCs are orthogonal by construction)")

## 7. Extreme individuals

Who sits furthest out, in SD units, and whether the same people are extreme in
more than one fit. A person extreme in every fit is a property of that person; a
person extreme in one is a property of that variant set.

In [ ]:
z_rows = []
for k in FITS_PANEL:
    d = E[k]
    z = (d - d.mean()) / d.std()
    mx = z.abs().max(axis=1)
    for pid, val in mx.nlargest(15).items():
        z_rows.append({"fit": k, "person_id": pid, "max_abs_z": val,
                       "on_PC": int(z.loc[pid].abs().idxmax()[2:])})
ext = pd.DataFrame(z_rows)
print(ext.round(2).to_string(index=False))

if len(ext):
    counts = ext["person_id"].value_counts()
    rep = counts[counts > 1]
    print(f"\nindividuals extreme in more than one fit: {len(rep)}")
    if len(rep):
        print(rep.to_string())

In [ ]:
fig, axes = plt.subplots(1, len(FITS_PANEL), figsize=(4 * len(FITS_PANEL), 3.6),
                         squeeze=False)
for ax, k in zip(axes[0], FITS_PANEL):
    d = E[k]
    z = ((d - d.mean()) / d.std()).abs().max(axis=1)
    ax.hist(z, bins=120, log=True)
    ax.axvline(5, color="crimson", ls="--", lw=1, label="5 SD")
    ax.set_xlabel("max |z| over PCs"); ax.set_ylabel("individuals (log)")
    ax.set_title(k.split("/", 1)[1], fontsize=8); ax.legend(fontsize=7)
plt.suptitle("How heavy is the individual-level tail?")
plt.tight_layout(); plt.savefig(f"{OUT}/07_extremity.png", dpi=150); plt.show()

## Save and upload

In [ ]:
TOP_TSV = f"{OUT}/top_loadings.tsv"
XF_TSV  = f"{OUT}/cross_fit_correlations.tsv"
EXT_TSV = f"{OUT}/extreme_individuals.tsv"
top.to_csv(TOP_TSV, sep="\t", index=False, float_format="%.6f")
xf.to_csv(XF_TSV, sep="\t", index=False, float_format="%.5f")
ext.to_csv(EXT_TSV, sep="\t", index=False, float_format="%.3f")
print("\n".join(f"wrote {p}" for p in (TOP_TSV, XF_TSV, EXT_TSV)))
print(f"figures: {len(os.listdir(OUT))} files in {OUT}")

subprocess.run(["bash", "-c", f"""
set -eo pipefail
gcloud storage cp "{OUT}/"*.tsv "{OUT_GS}/"
gcloud storage cp "{OUT}/"*.png "{OUT_GS}/figures/"
"""], check=True)
print(f"uploaded to {OUT_GS}")

## Copy notebook to bucket

In [ ]:
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/extra/rare_pca_plots.ipynb')
_gs = f'{OUT_GS}/notebooks/rare_pca_plots.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')